# Process

##### Steps: 
1. read the column
2. define schema in StructType and StructField 
3. parse the schema using from_json function 
4. explode the parsed cloumn
5. combine the code
6. write it into silver schema

In [0]:

df = spark.read.table("ecommerce_analytics.bronze.sales")
df.display()

In [0]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import *
from delta.tables import DeltaTable

##########################################
############--- silver_sales ---#############
##########################################

def silver_sales (df):
    product_schema = StructType([
        StructField('id', StringType(), True),
        StructField('name', StringType(), True),
        StructField('price', FloatType(), True),
        StructField('curr', StringType(), True),
        StructField('qty', IntegerType(), True),
        StructField('unit', StringType(), True)
    ])

    ''' # Another way to define schema of nested columns using _parse_datatype_string 
    
    from pyspark.sql.types import _parse_datatype_string as p

    schema = "curr int, id int, name string, price double, qty int, unit string"
    schema = p(schema)
    print(schema)
    '''

    df_sales = df.withColumn('product', from_json('product', product_schema)) \
                    .select('customer_id', 'customer_name', 'product_name', 'order_date', 'product_category', 'product.*', 'total_price')
    return df_sales
# sales_df.display()
# sales_df.write.mode('overwrite').saveAsTable('ecommerce_analytics.silver.silver_sales')

##################################
#####--SCD-1 IMPLEMENTATION--#####
##################################

def scd_merge_table(spark, source_table, target_table, business_key):

    # source_df = spark.table(source_table)

    if not spark.catalog.tableExists(target_table):
        print("First Load: Creating Silver Table :", target_table)
        source_table.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print("First Load: Silver Table Created : ", target_table)

    else:
        print("Incremental Load: Performing SCD type 1 Merge on ", target_table) 
        delta_table = DeltaTable.forName(spark, target_table)

        merge_condition = "AND".join(
            [f"target.{col} = source.{col}" for col in business_key]
        )
        ## ouptput of above merge_condition:
        ## e.g. ["target.customer_id = source.customer_id" and "target.product_id = source.product_id" etc.]

        delta_table.alias("target").merge(source_table.alias("source"),merge_condition,)\
                                        .whenMatchedUpdateAll()\
                                        .whenNotMatchedInsertAll()\
                                        .execute()
        print("Incremental Load: SCD type 1 Merge Completed on ", target_table)


##########################################
############--- MAIN CODE ---#############
##########################################

bronze_table_sales = 'ecommerce_analytics.bronze.sales'
silver_table_sales = 'ecommerce_analytics.silver.silver_sales'
business_key_sales = ['customer_id', 'id']

print(f"Reading bronze layer table: {bronze_table_sales}")
df = spark.read.table(bronze_table_sales)

print(f"Transforming bronze layer table: {bronze_table_sales} to silver layer table: {silver_table_sales}")
sales_df = silver_sales(df)

print(f"Performing SCD type 1 Merge on silver layer table: {silver_table_sales}")
scd_merge_table(spark, sales_df, silver_table_sales, business_key_sales)

print(f"Finished: Write completed for all silver layer tables")
